# 📈 中芯国际（688981.SH）近一年股票数据分析

**数据区间：** 2025-07-01 至 2026-07-01  
**数据来源：** Tushare / 腾讯自选股行情数据  
**分析内容：** K线图、移动均线、成交量、收益分析

---

In [ ]:
# 导入所需库
import pandas as pd
import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import plotly.express as px

# 设置中文显示
import plotly.io as pio
pio.templates.default = 'plotly_white'

print('✅ 库加载完成')

## 1. 数据加载与预览

In [ ]:
# 读取 CSV 数据
df = pd.read_csv('中芯国际_688981_近一年数据.csv')

# 转换日期格式
df['date'] = pd.to_datetime(df['date'])
df = df.sort_values('date').reset_index(drop=True)

# 查看基本信息
print(f'📊 数据概览：')
print(f'   交易日数：{len(df)} 天')
print(f'   起始日期：{df["date"].min().strftime("%Y-%m-%d")}')
print(f'   结束日期：{df["date"].max().strftime("%Y-%m-%d")}')
print(f'   数据列：{", ".join(df.columns.tolist())}')
print()

# 前5行数据
df.head()

## 2. 数据统计概览

In [ ]:
# 关键统计指标
start_price = df.iloc[0]['close']
end_price = df.iloc[-1]['close']
total_return = (end_price - start_price) / start_price * 100
max_price = df['high'].max()
min_price = df['low'].min()
avg_volume = df['volume'].mean()

print('📊 中芯国际 688981 关键指标')
print('=' * 50)
print(f'区间起始收盘价：   ¥{start_price:.2f}')
print(f'区间结束收盘价：   ¥{end_price:.2f}')
print(f'区间总收益率：     {total_return:+.2f}%')
print(f'区间最高价：       ¥{max_price:.2f}（{df.loc[df["high"].idxmax(), "date"].strftime("%Y-%m-%d")}）')
print(f'区间最低价：       ¥{min_price:.2f}（{df.loc[df["low"].idxmin(), "date"].strftime("%Y-%m-%d")}）')
print(f'日均成交量：       {avg_volume/1e6:.1f} 百万股')
print(f'日均成交额：       ¥{df["amount"].mean()/1e8:.1f} 亿元')
print(f'涨跌天数：         📈 上涨 {len(df[df["close"] >= df["open"]])} 天 / 📉 下跌 {len(df[df["close"] < df["open"]])} 天')

# 数据描述性统计
print('\n📋 描述性统计：')
df[['open', 'high', 'low', 'close', 'volume', 'amount']].describe().round(2)

## 3. 计算移动均线 (MA)

In [ ]:
# 计算多周期移动均线
df['MA5'] = df['close'].rolling(window=5).mean()
df['MA10'] = df['close'].rolling(window=10).mean()
df['MA20'] = df['close'].rolling(window=20).mean()
df['MA60'] = df['close'].rolling(window=60).mean()

# 计算日收益率
df['daily_return'] = df['close'].pct_change() * 100

# 显示最新20日均线数据
df[['date', 'close', 'MA5', 'MA10', 'MA20', 'MA60']].tail(10).round(2)

## 4. 🕯️ K线图 + 移动均线 + 成交量

使用 Plotly 绘制交互式K线蜡烛图，叠加MA均线，下方展示成交量。

In [ ]:
# 创建双面板图表
fig = make_subplots(
    rows=2, cols=1,
    shared_xaxes=True,
    vertical_spacing=0.03,
    row_heights=[0.7, 0.3],
    subplot_titles=('K线图 + 移动均线', '成交量')
)

# --- 上行：K线蜡烛图 ---
fig.add_trace(
    go.Candlestick(
        x=df['date'],
        open=df['open'],
        high=df['high'],
        low=df['low'],
        close=df['close'],
        name='K线',
        showlegend=True,
        increasing=dict(line=dict(color='#ef5350', width=1), fillcolor='#ef5350'),
        decreasing=dict(line=dict(color='#26a69a', width=1), fillcolor='#26a69a')
    ),
    row=1, col=1
)

# 添加上涨/下跌背景色带来区分趋势
up_days = df[df['close'] >= df['open']]
down_days = df[df['close'] < df['open']]

# --- 上行：移动均线 ---
ma_colors = {'MA5': '#f9a825', 'MA10': '#ff7043', 'MA20': '#ab47bc', 'MA60': '#42a5f5'}
for ma_name, color in ma_colors.items():
    fig.add_trace(
        go.Scatter(
            x=df['date'],
            y=df[ma_name],
            mode='lines',
            name=ma_name,
            line=dict(color=color, width=1.2),
            opacity=0.85
        ),
        row=1, col=1
    )

# --- 下行：成交量柱状图 ---
colors_volume = ['#ef5350' if close >= open_ else '#26a69a' 
                 for close, open_ in zip(df['close'], df['open'])]

fig.add_trace(
    go.Bar(
        x=df['date'],
        y=df['volume'],
        name='成交量',
        marker=dict(color=colors_volume, opacity=0.7),
        showlegend=True
    ),
    row=2, col=1
)

# --- 布局设置 ---
fig.update_layout(
    title=dict(
        text=f'中芯国际 688981.SH 近一年K线图<br><sup>{df["date"].min().strftime("%Y-%m-%d")} ~ {df["date"].max().strftime("%Y-%m-%d")} | 区间收益率：{total_return:+.2f}%</sup>',
        font=dict(size=20),
        x=0.5
    ),
    xaxis_rangeslider_visible=False,
    height=750,
    hovermode='x unified',
    legend=dict(
        orientation='h',
        yanchor='bottom',
        y=-0.25,
        xanchor='center',
        x=0.5
    ),
    margin=dict(l=60, r=30, t=80, b=60)
)

# X轴格式
fig.update_xaxes(
    rangeslider_visible=False,
    showgrid=True, gridwidth=0.5, gridcolor='#e0e0e0'
)

# Y轴格式
fig.update_yaxes(title_text='价格 (¥)', row=1, col=1, showgrid=True, gridwidth=0.5, gridcolor='#e0e0e0')
fig.update_yaxes(title_text='成交量 (股)', row=2, col=1, showgrid=True, gridwidth=0.5, gridcolor='#e0e0e0')

fig.show()

print(f'\n📌 区间总收益率：{total_return:+.2f}%')
print(f'📌 最高价 ¥{max_price:.2f}，最低价 ¥{min_price:.2f}')
print(f'📌 图表为交互式：可缩放、拖拽、悬停查看详情')

## 5. 📊 月度收益分析

In [ ]:
# 计算月度涨跌幅
df['year_month'] = df['date'].dt.to_period('M')
monthly = df.groupby('year_month').agg(
    月初开盘=('open', 'first'),
    月末收盘=('close', 'last'),
    最高价=('high', 'max'),
    最低价=('low', 'min'),
    月成交量=('volume', 'sum'),
    月成交额=('amount', 'sum')
).reset_index()

monthly['月度涨跌幅'] = ((monthly['月末收盘'] - monthly['月初开盘']) / monthly['月初开盘'] * 100).round(2)
monthly['year_month'] = monthly['year_month'].astype(str)

# 涨跌幅颜色
colors_month = ['#ef5350' if v > 0 else '#26a69a' for v in monthly['月度涨跌幅']]

fig_month = go.Figure()
fig_month.add_trace(go.Bar(
    x=monthly['year_month'],
    y=monthly['月度涨跌幅'],
    marker=dict(color=colors_month),
    text=[f'{v:+.2f}%' for v in monthly['月度涨跌幅']],
    textposition='outside',
    textfont=dict(size=11),
    name='月度涨跌幅'
))

fig_month.update_layout(
    title='中芯国际 月度涨跌幅',
    height=400,
    yaxis_title='涨跌幅 (%)',
    xaxis_title='月份',
    hovermode='x unified',
    showlegend=False
)
fig_month.update_yaxes(zeroline=True, zerolinecolor='#333', zerolinewidth=1.5)

fig_month.show()

# 打印统计
up_months = (monthly['月度涨跌幅'] > 0).sum()
print(f'📈 上涨月份：{up_months} / {len(monthly)} 个月')
print(f'📉 下跌月份：{len(monthly) - up_months} / {len(monthly)} 个月')
print(f'🏆 涨幅最大月：{monthly.loc[monthly["月度涨跌幅"].idxmax(), "year_month"]}（{monthly["月度涨跌幅"].max():+.2f}%）')
print(f'💥 跌幅最大月：{monthly.loc[monthly["月度涨跌幅"].idxmin(), "year_month"]}（{monthly["月度涨跌幅"].min():+.2f}%）')

## 6. 📈 价格走势与日均线交叉分析

In [ ]:
# MA5 与 MA20 交叉信号
df['signal'] = 0
df.loc[df['MA5'] > df['MA20'], 'signal'] = 1
df['cross'] = df['signal'].diff()

golden_cross = df[df['cross'] == 1]   # 金叉（MA5上穿MA20）
death_cross = df[df['cross'] == -1]    # 死叉（MA5下穿MA20）

fig_cross = go.Figure()

# 收盘价走势
fig_cross.add_trace(go.Scatter(
    x=df['date'], y=df['close'],
    mode='lines', name='收盘价',
    line=dict(color='#333', width=1.5)
))

# 均线
fig_cross.add_trace(go.Scatter(
    x=df['date'], y=df['MA5'],
    mode='lines', name='MA5',
    line=dict(color='#f9a825', width=1.2, dash='dot')
))
fig_cross.add_trace(go.Scatter(
    x=df['date'], y=df['MA20'],
    mode='lines', name='MA20',
    line=dict(color='#ab47bc', width=1.2, dash='dot')
))

# 金叉标记
fig_cross.add_trace(go.Scatter(
    x=golden_cross['date'], y=golden_cross['close'],
    mode='markers', name='金叉 (买入信号)',
    marker=dict(symbol='triangle-up', size=12, color='#ef5350', line=dict(width=1, color='darkred'))
))

# 死叉标记
fig_cross.add_trace(go.Scatter(
    x=death_cross['date'], y=death_cross['close'],
    mode='markers', name='死叉 (卖出信号)',
    marker=dict(symbol='triangle-down', size=12, color='#26a69a', line=dict(width=1, color='darkgreen'))
))

fig_cross.update_layout(
    title='MA5 与 MA20 金叉/死叉信号',
    height=500,
    hovermode='x unified',
    yaxis_title='价格 (¥)'
)

fig_cross.show()

print(f'🔴 金叉信号（买入）：{len(golden_cross)} 次')
print(f'🟢 死叉信号（卖出）：{len(death_cross)} 次')

## 7. 📉 日收益率分布

In [ ]:
# 日收益率直方图
fig_hist = go.Figure()

fig_hist.add_trace(go.Histogram(
    x=df['daily_return'].dropna(),
    nbinsx=40,
    marker=dict(
        color='#42a5f5',
        line=dict(color='#1e88e5', width=1)
    ),
    name='日收益率分布'
))

# 添加均值线
mean_ret = df['daily_return'].mean()
fig_hist.add_vline(x=mean_ret, line_dash='dash', line_color='#ef5350',
                   annotation_text=f'均值 {mean_ret:.3f}%', annotation_position='top right')

fig_hist.update_layout(
    title='日收益率分布直方图',
    height=400,
    xaxis_title='日收益率 (%)',
    yaxis_title='频次',
    bargap=0.05
)

fig_hist.show()

# 统计
ret = df['daily_return'].dropna()
print(f'📊 日收益率统计：')
print(f'   均值：{ret.mean():.4f}%')
print(f'   标准差：{ret.std():.4f}%')
print(f'   最大值：{ret.max():.4f}%（{df.loc[ret.idxmax(), "date"].strftime("%Y-%m-%d")}）')
print(f'   最小值：{ret.min():.4f}%（{df.loc[ret.idxmin(), "date"].strftime("%Y-%m-%d")}）')
print(f'   偏度：{ret.skew():.4f}')
print(f'   峰度：{ret.kurtosis():.4f}')

## 8. 📦 总结

通过以上分析，我们完成了对中芯国际（688981.SH）近一年数据的完整探索：

- **数据加载与清洗**：241个交易日完整日线数据
- **K线图可视化**：交互式蜡烛图 + MA5/MA10/MA20/MA60 均线 + 成交量
- **月度收益分析**：按月统计涨跌幅分布
- **均线交叉信号**：MA5与MA20的金叉/死叉买卖信号识别
- **日收益率分布**：统计分析收益的分布特征

> 💡 提示：本笔记本中的所有图表均为交互式，可以进行缩放、平移、悬停查看数据详情等操作。